# 02 · Analysing the results

**The thesis: the CSVs are raw; the paper needs tables and a figure, split by
question family, with a spread.**

No API key needed. This notebook only reads files.

## What this notebook demonstrates

| Name | What it does | Example |
|---|---|---|
| `load the CSVs` | read every result file into one table | `baseline.csv`, `agent.csv` |
| `accuracy by family` | the Lookup vs Compute split the paper reports | 89% vs 68% |
| `mean +/- spread` | summarise across repeats | 78.0% +/- 4.3% |
| `the figure` | accuracy by family, one bar group per configuration | `results/accuracy.png` |

## Step 1 — load everything that exists

In [1]:
import glob, os
import pandas as pd

paths = sorted(glob.glob("results-input/*.csv")) + \
        sorted(glob.glob("../rashan-table-agent/results/*.csv"))
print("found:", [os.path.basename(p) for p in paths])
df = pd.concat([pd.read_csv(p) for p in paths], ignore_index=True)
print("rows:", len(df), "| configs:", sorted(df["config"].unique()))

found: ['chatgpt_reference_sample.csv', 'agent.csv', 'baseline.csv', 'cot.csv']
rows: 340 | configs: ['agent', 'baseline', 'chatgpt_reference', 'cot']


## Step 2 — overall accuracy per configuration

In [2]:
acc = df.groupby("config")["correct"].agg(["mean", "count"])
acc["accuracy"] = (acc["mean"] * 100).round(1)
print(acc[["accuracy", "count"]].to_string())

                   accuracy  count
config                            
agent                  77.0    100
baseline               78.0    100
chatgpt_reference      65.0     40
cot                    79.0    100


## Step 3 — the split that matters

Never report a blended number. A system can be excellent at finding a cell and
hopeless at computing with it, and the blend hides that.

In [3]:
if "task_family" in df.columns and df["task_family"].notna().any():
    piv = df.pivot_table(index="config", columns="task_family",
                         values="correct", aggfunc="mean") * 100
    print(piv.round(1).to_string())
else:
    print("These CSVs predate the task_family column - rerun to get the split.")

task_family  Compute  Lookup
config                      
agent           67.9    87.2
baseline        67.9    89.4
cot             69.8    89.4


## Step 4 — spread across repeats

If the CSVs contain a `run` column, each repeat is a separate measurement.
Report the mean and the spread. **If two configurations differ by less than the
spread, you have not found a difference.**

In [4]:
if "run" in df.columns and df["run"].nunique() > 1:
    per_run = df.groupby(["config", "run"])["correct"].mean() * 100
    s = per_run.groupby("config").agg(["mean", "std", "count"]).round(1)
    print(s.to_string())
    print()
    print("Rule: a gap smaller than 'std' is not a result.")
else:
    print("Only one run present - no spread can be computed yet.")
    print("Run with --repeats 3 before claiming any difference.")

Only one run present - no spread can be computed yet.
Run with --repeats 3 before claiming any difference.


## Step 5 — the figure

In [5]:
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt

os.makedirs("results", exist_ok=True)
if "task_family" in df.columns and df["task_family"].notna().any():
    piv = (df.pivot_table(index="task_family", columns="config",
                          values="correct", aggfunc="mean") * 100)
    ax = piv.plot(kind="bar", figsize=(7, 4), edgecolor="white", linewidth=1.2)
    ax.set_ylabel("answered correctly (%)"); ax.set_ylim(0, 100)
    ax.set_xlabel(""); ax.legend(frameon=False, ncol=3)
    ax.grid(axis="y", alpha=.3); ax.set_axisbelow(True)
    for s in ("top", "right", "left"): ax.spines[s].set_visible(False)
    plt.tight_layout(); plt.savefig("results/accuracy_by_family.png", dpi=130)
    print("saved results/accuracy_by_family.png")
    print(piv.round(1).to_string())
else:
    print("no task_family column yet - skipping the figure")

saved results/accuracy_by_family.png
config       agent  baseline   cot
task_family                       
Compute       67.9      67.9  69.8
Lookup        87.2      89.4  89.4


## Step 6 — write down what you see

In `results/NOTES.md`, five to eight sentences:

```
 · which configuration was best, and by how much
 · is that gap bigger than the spread?  if not, say "no difference"
 · where is the system weakest - Lookup or Compute?
 · anything that surprised you
```

Plain sentences. If the answer is "nothing helped", write exactly that.

## What to carry out of this notebook

```
 · always report Lookup and Compute separately
 · always report the spread, and never claim a gap smaller than it
 · the figure is a summary of the table, not a replacement for it
```

Next: `03_error_analysis.ipynb` — why the wrong answers were wrong.